In [ ]:
%gui qt
%load_ext autoreload
%autoreload 2

In [ ]:
import hmt_v3 as hmt
import numpy as np
import pandas as pd

In [ ]:
me3_raw_df = pd.read_csv("test_data/k27_k27_thaw009_me3.csv")
ac_raw_df = pd.read_csv("test_data/k27_k27_thaw009_ac.csv")

me3_filtered_df = hmt.preprocess.filter_axial(me3_raw_df)
ac_filtered_df = hmt.preprocess.filter_axial(ac_raw_df)

binary_mask, me3_df, ac_df = hmt.preprocess.binarize_nucleus(me3_filtered_df, ac_filtered_df, thresh=2, bin_size=50)
distance_map, contour_bands = hmt.preprocess.create_radial_contours(binary_mask, show_plots=True)

In [ ]:
step=10

print("Extracting emperical H3K27me3 distributions...")
me3_rdf, me3_adf = hmt.simulate.extract_empirical_parameters(me3_df, sdis=500, step=step)

print("Extracting emperical H3K27ac distributions...")
ac_rdf, ac_adf = hmt.simulate.extract_empirical_parameters(ac_df, sdis=500, step=step)

In [ ]:
hmt.visualize.plot_rdf_adf(me3_rdf, me3_adf, ac_rdf, ac_adf, step=10)

In [ ]:
me3_n = hmt.simulate.extract_n_locs_from_rdf(me3_rdf, step=step)
ac_n  = hmt.simulate.extract_n_locs_from_rdf(ac_rdf,  step=step)

print(f"Mean Number of Localizations per nanodomain: \nH3K27me3: {me3_n} \nH3K27ac: {ac_n}")

me3_seeds = [[0, 0, 0]]
me3_locs = hmt.simulate.spawn_nanodomains(me3_seeds, n_locs=me3_n, step=step, rdf=me3_rdf, adf=me3_rdf)

ac_seeds = [[900, 0, 0]]
ac_locs = hmt.simulate.spawn_nanodomains(ac_seeds, n_locs=ac_n, step=step, rdf=ac_rdf, adf=ac_rdf, start_label=len(me3_seeds))

seeds = np.concat([me3_seeds, ac_seeds])
locs = pd.concat([me3_locs, ac_locs], ignore_index=True)
hmt.visualize.plot_nanodomain_2d(locs, seeds)
hmt.visualize.plot_nanodomain_3d(locs, seeds)

In [ ]:
me3_df = hmt.simulate.assign_contour_bands(me3_df, contour_bands, bin_size=50)
ac_df  = hmt.simulate.assign_contour_bands(ac_df,  contour_bands, bin_size=50)

inner_me3 = me3_df[me3_df['contour_band'] <= 20]
outer_me3 = me3_df[me3_df['contour_band'] > 80 ]

inner_ac  = ac_df[ac_df['contour_band'] <= 20]
outer_ac  = ac_df[ac_df['contour_band'] > 80 ]

print("Extracting emperical H3K27me3 distributions...")
inner_me3_rdf, inner_me3_adf = hmt.simulate.extract_empirical_parameters(inner_me3, sdis=500, step=step)
outer_me3_rdf, outer_me3_adf = hmt.simulate.extract_empirical_parameters(outer_me3, sdis=500, step=step)

print("Extracting emperical H3K27ac distributions...")
inner_ac_rdf,  inner_ac_adf  = hmt.simulate.extract_empirical_parameters(inner_ac, sdis=500, step=step)
outer_ac_rdf,  outer_ac_adf  = hmt.simulate.extract_empirical_parameters(outer_ac, sdis=500, step=step)

In [ ]:
hmt.visualize.plot_rdf_adf(inner_me3_rdf, inner_me3_adf, inner_ac_rdf, inner_ac_adf)
hmt.visualize.plot_rdf_adf(outer_me3_rdf, outer_me3_adf, outer_ac_rdf, outer_ac_adf)

inner_me3_n = hmt.simulate.extract_n_locs_from_rdf(inner_me3_rdf, step=step)
inner_ac_n  = hmt.simulate.extract_n_locs_from_rdf(inner_ac_rdf,  step=step)
print(f"Mean Number of Localizations per nanodomain from inner rings: \nH3K27me3: {inner_me3_n} \nH3K27ac: {inner_ac_n}\n")

outer_me3_n = hmt.simulate.extract_n_locs_from_rdf(outer_me3_rdf, step=step)
outer_ac_n  = hmt.simulate.extract_n_locs_from_rdf(outer_ac_rdf,  step=step)
print(f"Mean Number of Localizations per nanodomain from outer rings: \nH3K27me3: {outer_me3_n} \nH3K27ac: {outer_ac_n}")